In [2]:
from pyspark.sql import functions as F

# ============================================================
# NB_Gold_Fact_Payments
# Silver -> Gold
# Grain: one row per payment
# ============================================================

PAYMENTS_TABLE = "silver_payments"
ORDERS_TABLE = "silver_orders"

CUSTOMER_DIM = "dim_customer"
DATE_DIM = "dim_date"

TARGET_TABLE = "fact_payments"

# ------------------------------------------------------------
# 1. Read Silver tables
# ------------------------------------------------------------

payments = spark.table(PAYMENTS_TABLE)

orders = (
    spark.table(ORDERS_TABLE)
    .select(
        "order_id",
        "customer_id"
    )
)

# ------------------------------------------------------------
# 2. Read dimensions
# ------------------------------------------------------------

customers = (
    spark.table(CUSTOMER_DIM)
    .select(
        "customer_key",
        "customer_id"
    )
)

dates = (
    spark.table(DATE_DIM)
    .select(
        "date_key",
        "date"
    )
)

# ------------------------------------------------------------
# 3. Join payments with orders
# ------------------------------------------------------------

payments = (
    payments
    .join(
        orders,
        on="order_id",
        how="inner"
    )
)

# ------------------------------------------------------------
# 4. Join customer dimension
# ------------------------------------------------------------

payments = (
    payments
    .join(
        customers,
        on="customer_id",
        how="left"
    )
)

# ------------------------------------------------------------
# 5. Derive payment date
# ------------------------------------------------------------

payments = payments.withColumn(
    "payment_date_only",
    F.to_date("payment_date")
)

# ------------------------------------------------------------
# 6. Join date dimension
# ------------------------------------------------------------

payments = (
    payments
    .join(
        dates,
        payments.payment_date_only == dates.date,
        how="left"
    )
)

# ------------------------------------------------------------
# 7. Build Gold fact
# ------------------------------------------------------------

df_gold = (
    payments
    .select(
        "payment_id",
        "order_id",
        "customer_key",
        "date_key",
        "payment_date",
        "payment_method",
        "amount",
        "status"
    )
    .withColumn(
        "payment_count",
        F.lit(1)
    )
)

# ------------------------------------------------------------
# 8. Write Gold fact
# ------------------------------------------------------------

df_gold.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(TARGET_TABLE)

# ------------------------------------------------------------
# 9. Validation
# ------------------------------------------------------------

print(f"Gold table created: {TARGET_TABLE}")
print(f"Records: {df_gold.count()}")

df_gold.show(10, truncate=False)

StatementMeta(, 2d3f4e37-7371-4efb-b9cf-049ca798826b, 4, Finished, Available, Finished, False)

Gold table created: fact_payments
Records: 49978
+----------+--------+------------+--------+-------------------+--------------+-------+---------+-------------+
|payment_id|order_id|customer_key|date_key|payment_date       |payment_method|amount |status   |payment_count|
+----------+--------+------------+--------+-------------------+--------------+-------+---------+-------------+
|1         |26772   |6030        |20260810|2026-08-10 14:35:07|debit_card    |2601.82|completed|1            |
|2         |13398   |4579        |20250504|2025-05-04 09:11:07|paypal        |1277.31|completed|1            |
|3         |3747    |85          |20250707|2025-07-07 13:50:30|paypal        |3589.99|completed|1            |
|4         |26998   |2841        |20260721|2026-07-21 02:54:40|bank_transfer |612.93 |completed|1            |
|5         |18247   |7989        |20251105|2025-11-05 08:05:36|bank_transfer |571.68 |completed|1            |
|6         |40582   |4550        |20260711|2026-07-11 04:49:07|